# 教程 01：Ascend C 算子基础

**学习目标** ⭐
- 理解 Ascend C 算子的基本结构
- 掌握本仓库统一的算子目录规范
- 了解算子的构建和测试流程

---

## Ascend C 算子结构

本仓库中的每个 Ascend C 算子遵循统一目录结构：

```
<算子名>/
├── op_kernel/       # Device 侧 Kernel 代码 (Ascend C, __aicore__)
│   ├── <op>.cpp     # Kernel 实现
│   └── <op>.h       # Kernel 头文件
├── op_host/          # Host 侧调度代码
│   ├── <op>_host.h  # Host API 声明
│   ├── <op>_host.cpp # Host 实现 (Tiling + Launch)
│   └── <op>_def.cpp  # 算子定义 (aclnn API)
├── tests/            # 测试
│   ├── ut/op_kernel/ # C++ 单元测试 (GTest)
│   ├── test_<op>.py  # Python 集成测试
│   └── benchmark_<op>.py # 性能基准
└── examples/         # 使用示例
    └── test_aclnn_<op>.cpp
```

## 核心概念

### 1. Kernel (设备侧)
Kernel 是运行在 NPU 上的核心计算逻辑，使用 Ascend C 编写：

```cpp
__aicore__ void kernel_name(GM void* input, GM void* output, ...) {
    // 1. 数据搬运：GM → Local Memory
    // 2. 计算：在 AI Core 上执行
    // 3. 结果写回：Local Memory → GM
}
```

### 2. Host (主机侧)
Host 代码负责：
- 参数校验和 Tiling（将大任务切分为小块）
- 内存分配和准备
- 调用 Kernel 启动

### 3. aclnn API
对外提供统一的调用接口：`aclnn{OpName}(...)`

## 本仓库算子统计

| 领域 | 算子 | 类型 | 状态 |
|------|------|------|------|
| AI4MD | Lennard_Jones | Ascend C | ✅ 完整测试 |
| AI4MD | GAFF2 | Ascend C | ✅ 已发布 |
| AI4MD | PME | Ascend C | ✅ 已发布 |
| AI4MD | SHAKE | Ascend C | ✅ 已发布 |
| AI4MD | velocity-verlet | Ascend C | ✅ 已发布 |
| AI4MD | DPD | Ascend C | ✅ 完整测试 |
| AI4PDE | PINN | Ascend C | ✅ 完整测试 |
| AI4PDE | FNO | Ascend C | ✅ 完整测试 |
| AI4PDE | DeepONet | Ascend C | ✅ 完整测试 |
| AI4PDE | MeshGraphNet | Ascend C | ✅ 完整测试 |
| MatPred | DAO | PyTorch 参考 | 🔧 基础库 |
| Tabular | TabNet | PyTorch 参考 | 🔧 可运行 |
| TimeSeries | TimesNet | PyTorch 参考 | 🔧 可运行 |
| SmallData | GPR + BO | PyTorch 参考 | 🔧 可运行 |

In [ ]:
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
template_dir = os.path.join(repo_root, 'template')

template_files = os.listdir(template_dir)
print("模板文件:")
for f in sorted(template_files):
    fpath = os.path.join(template_dir, f)
    size = os.path.getsize(fpath)
    print(f"  📄 {f} ({size:,} bytes)")

In [ ]:
# 以 LJ 力场算子为例，查看其目录结构
lj_dir = os.path.join(repo_root, 'simulation', 'AI4MD', 'Lennard_Jones')

print("LJ 算子目录结构:")
for root, dirs, files in os.walk(lj_dir):
    level = root.replace(lj_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    sub = '  ' * (level + 1)
    for file in files:
        if not file.endswith('.pyc') and not file.endswith('.inc'):
            print(f"{sub}{file}")

## 构建流程

```bash
cd simulation/AI4MD/<op>
mkdir build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DSOC_VERSION=Ascend910B1
make -j$(nproc)
ctest --output-on-failure
```

> 注意：需要 CANN ≥ 7.0 和配套的 NPU 环境才能编译运行 Ascend C 算子。
> 如果没有 NPU 环境，可参考教程 05-07 使用 PyTorch 参考实现。

## 总结 📋

- Ascend C 算子 = Kernel（设备侧）+ Host（主机侧）
- 本仓库所有算子遵循统一目录规范
- 提供了完整的贡献模板（template/）
- 部分算子可在无 NPU 环境下通过 PyTorch 参考学习

## 下一步

选择感兴趣的领域继续学习：
- [教程 02：材料性质预测](02_MaterialPrediction.ipynb)
- [教程 03：机器学习分子动力学](03_AI4MD_Tutorial.ipynb)
- [教程 04：AI for PDE](04_AI4PDE_Tutorial.ipynb)